<a href="https://www.kaggle.com/code/avikdas567/measuring-progress-toward-agi-cognitive-abilities?scriptVersionId=307882249" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import random
import json
import torch
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# LOAD MODEL

MODEL_PATH = "/kaggle/input/models/google/gemma/transformers/2b-it/3"

import os
print("Model files:", os.listdir(MODEL_PATH))

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    dtype=torch.float16,
)

model.eval()

# TASK GENERATORS

def generate_standard_task():
    tasks = ["A", "B", "C", "D"]
    constraints = []

    before = random.sample(tasks, 2)
    constraints.append(f"{before[0]} must come before {before[1]}")

    na = random.sample(tasks, 2)
    constraints.append(f"{na[0]} cannot be adjacent to {na[1]}")

    fixed_task = random.choice(tasks)
    fixed_pos = random.randint(1, 4)
    constraints.append(f"{fixed_task} must be in position {fixed_pos}")

    prompt = f"""
You are solving a planning problem.

Tasks: A, B, C, D

Constraints:
- {constraints[0]}
- {constraints[1]}
- {constraints[2]}

Provide:
1. A valid ordering of tasks (example format: A B C D)
2. A confidence score (0-100)

Answer in this format:
Answer: A B C D
Confidence: 85
"""

    return constraints, prompt.strip()


def generate_rule_switch_task():
    tasks = ["A", "B", "C", "D"]

    before = random.sample(tasks, 2)
    na = random.sample(tasks, 2)

    constraints = [
        f"{before[0]} must come before {before[1]}",
        f"{na[0]} cannot be adjacent to {na[1]}"
    ]

    new_rule = f"{before[1]} must come before {before[0]}"

    prompt = f"""
You are solving a planning problem.

Tasks: A, B, C, D

Initial Constraints:
- {constraints[0]}
- {constraints[1]}

UPDATED RULE (this overrides previous rules):
- {new_rule}

Provide:
1. A valid ordering of tasks (example format: A B C D)
2. A confidence score (0-100)

Answer in this format:
Answer: A B C D
Confidence: 85
"""

    return [new_rule, constraints[1]], prompt.strip()

# VALIDATION LOGIC

def check_valid(order, constraints):
    try:
        order = order.split()
        if len(order) != 4:
            return False

        pos = {task: i for i, task in enumerate(order)}

        for c in constraints:
            if "must come before" in c:
                a, b = c.split(" must come before ")
                if pos[a] >= pos[b]:
                    return False

            elif "cannot be adjacent" in c:
                a, b = c.split(" cannot be adjacent to ")
                if abs(pos[a] - pos[b]) == 1:
                    return False

        return True
    except:
        return False

# METACOGNITION SCORING

def calibration_score(conf, correct):
    conf = max(0, min(conf, 100))
    return (conf / 100) if correct else (1 - conf / 100)

# GENERATION

def generate_response(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# RUN BENCHMARK

N_TASKS = 40

results = []

for i in tqdm(range(N_TASKS)):

    if i < int(0.7 * N_TASKS):
        constraints, prompt = generate_standard_task()
        task_type = "standard"
    else:
        constraints, prompt = generate_rule_switch_task()
        task_type = "rule_switch"

    output = generate_response(prompt)

    answer = ""
    confidence = 50

    for line in output.split("\n"):
        line_clean = line.strip().lower()

        # FIX: more flexible parsing
        if "answer" in line_clean:
            answer = line.split(":", 1)[-1].strip().upper()

        if "confidence" in line_clean:
            try:
                confidence = int(line.split(":", 1)[-1].strip())
            except:
                confidence = 50

    correct = check_valid(answer, constraints)
    calib = calibration_score(confidence, correct)

    results.append({
        "task_type": task_type,
        "prompt": prompt,
        "constraints": constraints,
        "model_output": output,
        "parsed_answer": answer,
        "confidence": confidence,
        "correct": int(correct),
        "calibration_score": calib
    })

# FINAL METRICS

accuracy = np.mean([r["correct"] for r in results])
calibration = np.mean([r["calibration_score"] for r in results])

high_conf_wrong = [
    r for r in results if r["correct"] == 0 and r["confidence"] > 70
]

error_rate = len(high_conf_wrong) / len(results)

print("\nBENCHMARK RESULTS:-")
print(f"Accuracy: {accuracy:.3f}")
print(f"Calibration Score: {calibration:.3f}")
print(f"High-Confidence Error Rate: {error_rate:.3f}")

# SAVE OUTPUT

output_path = "/kaggle/working/benchmark_results.json"

with open(output_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"\nResults saved to {output_path}")

# FAILURE ANALYSIS

print("\nHigh Confidence Errors:", len(high_conf_wrong))

if len(high_conf_wrong) > 0:
    print("\nExample Failure Case:\n")
    print(high_conf_wrong[0]["model_output"])

Model files: ['model.safetensors.index.json', 'gemma-2b-it.gguf', 'config.json', 'model-00001-of-00002.safetensors', 'model-00002-of-00002.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json', '.gitattributes', 'tokenizer.model', 'generation_config.json']


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

100%|██████████| 40/40 [01:46<00:00,  2.67s/it]


BENCHMARK RESULTS:-
Accuracy: 0.350
Calibration Score: 0.483
High-Confidence Error Rate: 0.050

Results saved to /kaggle/working/benchmark_results.json

High Confidence Errors: 2

Example Failure Case:

You are solving a planning problem.

Tasks: A, B, C, D

Constraints:
- B must come before C
- A cannot be adjacent to B
- A must be in position 1

Provide:
1. A valid ordering of tasks (example format: A B C D)
2. A confidence score (0-100)

Answer in this format:
Answer: A B C D
Confidence: 85

**Explanation:**

* Task A must be completed first, as it needs to be in position 1.
* Task B must be completed before task C, as it must come before C.
* Task A cannot be adjacent to task B, as this would create an invalid configuration.
* Task A must be completed before task D, as it needs to be in position 1
